## Day 12 – Methodological Design

### Theoretical Foundation and Method Choice
*Focus: principled justification aligned with the use case*

> This investigation applies the principle of **linear convolution** as a spatial-domain filtering operation and **Wiener deconvolution** as a regularized inverse operation to the SMI daily price signal within the context of financial trend analysis in Switzerland.

**Convolution** is defined as:
$$y[n] = (x * h)[n] = \sum_{k=0}^{M-1} h[k] \cdot x[n-k]$$

where $x$ is the input signal, $h$ is the convolution kernel of length $M$, and $y$ is the filtered output.

**Kernel 1 – Moving Average (Uniform kernel):**
$$h_{\text{MA}}[k] = \frac{1}{M} \quad \text{for } k = 0, \ldots, M-1$$

Chosen because it provides equal-weight smoothing, appropriate for removing short-term noise without imposing shape assumptions on the trend.

**Kernel 2 – Gaussian kernel:**
$$h_{\text{G}}[k] = \frac{1}{\sqrt{2\pi}\sigma} \exp\left(-\frac{(k - \lfloor M/2 \rfloor)^2}{2\sigma^2}\right)$$

Chosen because it provides smoothing with a Gaussian weighting profile, preserving signal smoothness and having well-defined frequency-domain characteristics.

**Deconvolution method – Wiener filtering:**
Wiener deconvolution was selected because it provides a regularized inverse filter that balances reconstruction fidelity against noise amplification. Unlike naive inverse filtering ($Y/H$ in the frequency domain), Wiener filtering introduces a regularization parameter (noise-to-signal ratio) that prevents instability when the kernel's frequency response approaches zero.

> This principle assumes linearity and shift-invariance, and is valid under the assumption that the filtering operation is accurately modeled by convolution with a known kernel. If the kernel is misspecified or the noise level is underestimated, deconvolution may amplify noise or produce ringing artifacts, compromising the quality of the restored signal in this financial monitoring application.

### Parameter Definition and Mathematical Specification
*Focus: explicit parameter selection, derivation, and unit consistency*

> The signal has N ≈ 250 daily samples expressed in CHF.

**Convolution parameters:**
- Moving average kernel size: $M \in \{3, 5, 10, 20\}$ (trading days), where $M=5$ corresponds to 1 trading week
- Gaussian kernel: $\sigma = M / 6$ (so that the kernel covers approximately $\pm 3\sigma$ within the window)
- Boundary handling: zero-padding (extend signal with zeros at boundaries)

**Deconvolution parameters:**
- Wiener filter regularization parameter: will be varied as $\lambda \in \{0.001, 0.01, 0.1, 1.0\}$
- Baseline kernel: Moving average with $M=5$

### Experimental Design for Next Days
*Focus: structured parameter variation and theoretical prediction*

> The baseline configuration is defined as: moving average convolution with $M = 5$, Wiener deconvolution with $\lambda = 0.01$.

The following parameters will be varied systematically:

| Parameter | Range | Justification |
|-----------|-------|---------------|
| Kernel size M | 3, 5, 10, 20 | Tests smoothing strength from mild to aggressive |
| Kernel type | Moving Average, Gaussian | Compares uniform vs. weighted smoothing |
| Wiener regularization λ | 0.001, 0.01, 0.1, 1.0 | Tests deconvolution stability/fidelity trade-off |

> It is theoretically expected that larger kernel sizes produce smoother outputs with greater latency and peak attenuation. Gaussian kernels should produce smoother transitions than uniform kernels. Lower λ values produce sharper reconstruction but with increased noise amplification, while higher λ values yield more stable but blurred reconstructions.

### Methodological Limitations and Risk Factors
*Focus: assumptions, stability, and potential misinterpretation*

> This approach assumes that the noise component in the SMI signal can be approximated as additive white noise and that the smoothing kernel accurately represents the filtering operation, which may be violated when market microstructure effects or non-linear dynamics are present.  
> The method is expected to be reliable when the signal-to-noise ratio is moderate and the kernel size is small relative to the dominant signal features, but may become unstable when the Wiener regularization parameter is set too low, causing severe noise amplification.  
> In this financial monitoring use case, the primary risk factors are non-white noise characteristics and over-smoothing of transient events, potentially leading to delayed risk detection.